# T3chSeal: análisis estadístico y cálculo del sello

| Fase | Qué hace | Sección |
|---|---|---|
| 3 · Limpieza | Filtros, variables nuevas, Shorts y sorteos | 1 |
| 4 · EDA | Distribuciones, media frente a mediana, valores extremos | 2 |
| 5 · Tests estadísticos | Pearson, Spearman y Kruskal-Wallis | 3 y 4 |
| 6 · Score y validación | Percentiles por tier, sello, alertas, validación | 5 y 6 |

**Métodos usados:** Pearson, Spearman y Kruskal-Wallis.

In [4]:
# Librerías
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import os


# Subida de archivos a Colab (descomenta si hace falta)
from google.colab import files
files.upload()

# Cargamos los CSV generados por el script de extracción.
# sep=","  -> columnas separadas por comas
# encoding="utf-8-sig" -> lee bien tildes y emojis de los títulos
videos = pd.read_csv("videos_raw.csv", sep=",", encoding="utf-8-sig")    # 1 fila por vídeo
canales = pd.read_csv("channels_raw.csv", sep=",", encoding="utf-8-sig") # 1 fila por canal
print(len(videos), "vídeos |", len(canales), "canales")

ModuleNotFoundError: No module named 'google.colab'

## 1. Limpieza y transformación (fase 3)

In [ ]:
# FILTROS: quitamos los vídeos que distorsionan los resultados.
# - Menos de 100 vistas: sus porcentajes no son fiables
#   (5 comentarios en 20 vistas = 25 % de comment rate)
# - Menos de 7 días publicados: todavía no han acumulado sus vistas

videos["published_at"] = pd.to_datetime(videos["published_at"], utc=True)

# FECHA DE REFERENCIA para contar los días publicados.
# Antes usábamos "hoy", pero así el resultado cambia cada vez que se ejecuta.
# Usamos la fecha del vídeo más reciente (aprox. el día de la extracción).
# Si el número final no coincide con el del informe (19.228), escribe aquí
# la fecha de la primera ejecución, por ejemplo: pd.Timestamp("2026-11-02", tz="UTC")
FECHA_REFERENCIA = videos["published_at"].max()
videos["dias_publicado"] = (FECHA_REFERENCIA - videos["published_at"]).dt.days

antes = len(videos)
videos = videos[(videos["views"] >= 100) & (videos["dias_publicado"] >= 7)].copy()
print(f"Quitados {antes - len(videos)} vídeos; quedan {len(videos)}")

In [ ]:
# VARIABLES NUEVAS

# Comment rate: comentarios por cada 100 vistas (implicación de la audiencia)
videos["comment_rate"] = videos["comments"] / videos["views"] * 100

# Like rate: likes por cada 100 vistas (implicación ligera)
videos["like_rate"] = videos["likes"] / videos["views"] * 100

# Logaritmo de las vistas: comprime la cola larga de los virales.
# Sin él, un vídeo de 10 millones dominaría la correlación de Pearson.
videos["log_views"] = np.log10(videos["views"] + 1)

# Longitud del título en caracteres
videos["title_len"] = videos["title"].str.len()

# Sorteos: los detectamos por el título (la API no nos da las descripciones)
patron_sorteo = r"sorteo|giveaway|g[aá]nate|participa|concurso"
videos["es_sorteo"] = videos["title"].str.contains(patron_sorteo, case=False, na=False)

# Nombre del canal (el CSV de vídeos no lo trae): unimos por channel_id
if "channel_title" not in videos.columns:
    nombres = canales[["channel_id", "title"]].rename(columns={"title": "channel_title"})
    videos = videos.merge(nombres, on="channel_id", how="left")

print(videos["es_sorteo"].sum(), "vídeos de sorteo |",
      round(videos["is_short"].mean() * 100, 1), "% de Shorts")

## 2. Análisis exploratorio de datos, EDA (fase 4)

Antes de calcular nada miramos cómo son los datos. Lo que salga aquí decide los filtros y los métodos.

In [ ]:
# RESUMEN ESTADÍSTICO: número de datos, media, desviación, mínimo, cuartiles y máximo
videos[["views", "log_views", "comment_rate", "like_rate", "title_len"]].describe().round(3)

In [ ]:
# MEDIA FRENTE A MEDIANA: ¿por qué son tan distintas?
media = videos["views"].mean()
mediana = videos["views"].median()          # n par -> media de los dos vídeos centrales
media_geo = 10 ** np.log10(videos["views"]).mean()   # media en escala logarítmica

print(f"Media:            {media:,.0f}")
print(f"Mediana:          {mediana:,.0f}")
print(f"Media geométrica: {media_geo:,.0f}  (cerca de la mediana -> datos log-normales)")
print(f"Desv. típica / media: {videos['views'].std() / media:.1f}  (cola muy larga)")
print(f"Lo que sube la media el vídeo más visto, por sí solo: "
      f"{videos['views'].max() / len(videos):,.0f} vistas")

In [ ]:
# HISTOGRAMAS DE LA EDA
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

# A) Vistas en escala logarítmica, con media y mediana
ax[0].hist(np.log10(videos["views"]), bins=50, color="#5B7FA6")
ax[0].axvline(np.log10(mediana), color="black", lw=2, label=f"Mediana = {mediana:,.0f}")
ax[0].axvline(np.log10(media), color="#C2410C", lw=2, ls="--", label=f"Media = {media:,.0f}")
ax[0].set_xticks(range(2, 8))
ax[0].set_xticklabels(["100", "1K", "10K", "100K", "1M", "10M"])
ax[0].set_title("A. Vistas por vídeo (escala log.)")
ax[0].set_xlabel("Vistas"); ax[0].set_ylabel("Nº de vídeos"); ax[0].legend()

# B) Comment rate, con los sorteos en otro color
bins = np.linspace(0, 3, 61)
ax[1].hist(videos.loc[~videos["es_sorteo"], "comment_rate"].clip(upper=3), bins=bins,
           color="#5B7FA6", label="Resto de vídeos")
ax[1].hist(videos.loc[videos["es_sorteo"], "comment_rate"].clip(upper=3), bins=bins,
           color="#C2410C", label="Sorteos")
ax[1].set_yscale("log")
ax[1].set_title("B. Comment rate por vídeo (valores > 3 % agrupados en 3)")
ax[1].set_xlabel("Comentarios por cada 100 vistas"); ax[1].set_ylabel("Nº de vídeos (log.)")
ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# VALORES EXTREMOS: los 15 vídeos con mayor comment rate. ¿Son sorteos?
pd.set_option("display.max_colwidth", 80)
videos.nlargest(15, "comment_rate")[["channel_title", "title", "views", "comments", "comment_rate", "es_sorteo"]]

In [ ]:
# EFECTO DE LOS SORTEOS: mediana del comment rate con y sin sorteo
med = videos.groupby("es_sorteo")["comment_rate"].median()
print(med.round(3))
print(f"Los sorteos multiplican el comment rate por {med[True] / med[False]:.1f}")

**Conclusiones de la EDA**
- Las vistas tienen una cola muy larga: la media es unas 8 veces la mediana. Por eso trabajamos con medianas y métodos no paramétricos.
- Los sorteos (unos 45 vídeos) multiplican el comment rate por más de 6. Se excluyen del score.
- Cerca de la mitad de los vídeos son Shorts, con métricas muy distintas. El score se calcula solo con vídeos largos.

## 3. Correlaciones a nivel vídeo (fase 5, pregunta a)

**¿Qué métricas aportan información distinta?**
- Pearson mide la relación lineal y es sensible a los extremos.
- Spearman usa posiciones (1.º, 2.º…) y aguanta los extremos.
- Si los dos difieren mucho, hay valores extremos influyendo.

In [ ]:
numericas = ["log_views", "comment_rate", "like_rate", "title_len"]

# Tabla 1 del informe: Pearson y Spearman con su p-valor para cada pareja
parejas = [("log_views", "comment_rate"), ("comment_rate", "like_rate"),
           ("log_views", "like_rate"), ("title_len", "log_views"), ("title_len", "like_rate")]
filas = []
for a, b in parejas:
    d = videos[[a, b]].dropna()          # pearsonr no admite vacíos
    r, p_r = stats.pearsonr(d[a], d[b])
    rho, p_s = stats.spearmanr(d[a], d[b])
    filas.append({"pareja": f"{a} – {b}", "n": len(d), "pearson_r": round(r, 2), "p_pearson": p_r,
                  "spearman_rho": round(rho, 2), "p_spearman": p_s})
tabla1 = pd.DataFrame(filas)
tabla1

# Ojo: con casi 20.000 vídeos casi todo sale significativo (p < 0,001);
# importa más el TAMAÑO del coeficiente que el p-valor.

In [ ]:
# MAPA DE CALOR de Spearman (rojo = positiva, azul = negativa)
plt.figure(figsize=(6, 5))
sns.heatmap(videos[numericas].corr(method="spearman"), annot=True, fmt=".2f",
            cmap="RdBu_r", vmin=-1, vmax=1)
plt.title("Correlación de Spearman (nivel vídeo)")
plt.show()

## 4. Tabla de canales y comparación entre tiers (fase 5, pregunta b)

In [ ]:
# PREPARACIÓN PARA EL SCORE

# 1) Comentarios desactivados / likes ocultos -> ratio VACÍO (NaN), no 0.
#    Un canal con comentarios desactivados no tiene "0 % de implicación":
#    simplemente no se puede medir.
videos.loc[videos["comments_disabled"] == True, "comment_rate"] = np.nan
videos.loc[videos["likes_hidden"] == True, "like_rate"] = np.nan

# 2) Días entre vídeos consecutivos de cada canal, con decimales
videos = videos.sort_values(["channel_id", "published_at"])
videos["dias_entre"] = videos.groupby("channel_id")["published_at"].diff().dt.total_seconds() / 86400

# 3) Vídeos para el score: solo LARGOS y SIN sorteo  (~ significa "NO")
videos_score = videos[(videos["is_short"] == False) & (~videos["es_sorteo"])].copy()
print(len(videos_score), "vídeos largos sin sorteo para el score")

In [ ]:
# TABLA DE CANALES: 1 fila por canal (el sello se da a canales, no a vídeos)
por_canal = videos_score.groupby("channel_id").agg(
    canal=("channel_title", "first"),
    n_videos=("video_id", "count"),                         # vídeos largos analizados
    mediana_vistas=("views", "median"),                     # vistas del vídeo típico
    comment_rate=("comment_rate", "median"),                # implicación (comentarios)
    like_rate=("like_rate", "median"),                      # implicación (likes)
    cv_vistas=("views", lambda s: s.std() / s.mean()),      # estabilidad (bajo = predecible)
    media_dias_entre=("dias_entre", "mean"),                # cada cuántos días publica
    std_dias_entre=("dias_entre", "std"),                   # regularidad (bajo = constante)
).reset_index()

# % de Shorts y de sorteos: con TODOS los vídeos del canal (para las alertas)
extra = videos.groupby("channel_id").agg(
    pct_shorts=("is_short", "mean"),
    pct_sorteos=("es_sorteo", "mean"),
).reset_index()
por_canal = por_canal.merge(extra, on="channel_id", how="left")

# Datos del canal
por_canal = por_canal.merge(
    canales[["channel_id", "subscribers", "hidden_subscribers", "is_tech_topic"]],
    on="channel_id", how="left")

# Vistas por suscriptor (solo si los suscriptores son visibles)
por_canal["vistas_por_sub"] = np.where(
    por_canal["hidden_subscribers"] == True, np.nan,
    por_canal["mediana_vistas"] / por_canal["subscribers"])

# FILTROS: mínimo 20 vídeos largos (medianas fiables) y solo canales tech
antes = len(por_canal)
por_canal = por_canal[(por_canal["n_videos"] >= 20) & (por_canal["is_tech_topic"] == True)].copy()
print(f"Quitados {antes - len(por_canal)} canales; quedan {len(por_canal)}")

In [ ]:
# TIERS: 3 grupos del mismo tamaño según los suscriptores (terciles)
por_canal["tier"] = pd.qcut(por_canal["subscribers"], 3, labels=["bajo", "medio", "alto"])
por_canal.groupby("tier", observed=True)["subscribers"].agg(["min", "max", "count"])

In [ ]:
# KRUSKAL-WALLIS (Tabla 2 del informe): ¿cada métrica es distinta entre tiers?
# H0: los tres tiers tienen la misma distribución.
# p < 0,05 -> la métrica cambia con el tamaño -> hay que comparar dentro del tier
metricas = ["comment_rate", "like_rate", "vistas_por_sub", "mediana_vistas",
            "cv_vistas", "media_dias_entre", "std_dias_entre"]

resultados = []
for m in metricas:
    grupos = [g[m].dropna() for _, g in por_canal.groupby("tier", observed=True)]
    h, p = stats.kruskal(*grupos)
    medianas = por_canal.groupby("tier", observed=True)[m].median().round(3).to_dict()
    resultados.append({"metrica": m, **medianas, "H": round(h, 2), "p_valor": round(p, 4),
                       "diferente_entre_tiers": p < 0.05})
tabla2 = pd.DataFrame(resultados)
tabla2

In [ ]:
# CÓMO SE LLEGA AL p = 0,31 DEL COMMENT RATE (para explicarlo en la defensa)
d = por_canal.dropna(subset=["comment_rate"]).copy()
d["puesto"] = d["comment_rate"].rank()               # 1 = el que menos comenta
print("Puesto medio por tier (si no hubiera diferencias, todos rondarían",
      round(d["puesto"].mean(), 1), "):")
print(d.groupby("tier", observed=True)["puesto"].mean().round(1))

h, p = stats.kruskal(*[g["comment_rate"] for _, g in d.groupby("tier", observed=True)])
print(f"\nH = {h:.2f} | p = {p:.2f} | con 3 grupos, p = e^(-H/2) = {np.exp(-h / 2):.2f}")

# Prueba intuitiva: repartimos los canales en tiers AL AZAR 10.000 veces
rng = np.random.default_rng(1)
cr, t = d["comment_rate"].values, d["tier"].astype(str).values
veces = sum(stats.kruskal(*[cr[s == k] for k in ["bajo", "medio", "alto"]])[0] >= h
            for s in (rng.permutation(t) for _ in range(10000)))
print(f"En el {veces / 100:.0f} % de los repartos al azar sale una H igual o mayor")

In [ ]:
# TAMAÑO DEL CANAL FRENTE A IMPLICACIÓN (nivel canal, n = 115)
filas = []
for m in ["comment_rate", "like_rate", "vistas_por_sub", "mediana_vistas"]:
    d = por_canal[["subscribers", m]].dropna()
    rho, p = stats.spearmanr(d["subscribers"], d[m])
    filas.append({"metrica": m, "spearman_rho": round(rho, 2), "p_valor": round(p, 3)})
pd.DataFrame(filas)

In [ ]:
# SCATTER: suscriptores frente a comment rate y like rate, coloreado por tier
colores = {"bajo": "#8FB3D9", "medio": "#4A7DB5", "alto": "#1F3F66"}
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for i, m in enumerate(["comment_rate", "like_rate"]):
    d = por_canal[["subscribers", m, "tier"]].dropna()
    rho, p = stats.spearmanr(d["subscribers"], d[m])
    for tier, g in d.groupby("tier", observed=True):
        ax[i].scatter(g["subscribers"], g[m], color=colores[tier], label=f"Tier {tier}", s=30)
    # línea de tendencia sobre log10(suscriptores)
    x = np.log10(d["subscribers"]); a, b = np.polyfit(x, d[m], 1)
    xs = np.linspace(x.min(), x.max(), 50)
    ax[i].plot(10 ** xs, a * xs + b, color="#C2410C", lw=2)
    ax[i].set_xscale("log")
    ax[i].set_xlabel("Suscriptores (escala log.)"); ax[i].set_ylabel(f"{m} (%)")
    ax[i].set_title(f"{m}: Spearman ρ = {rho:.2f}; p = {p:.3f}")
ax[0].legend(); plt.tight_layout(); plt.show()

**Conclusiones de los tests**
- **Comment rate:** no cambia con el tamaño (p = 0,31; ρ = −0,05). Ser grande no es ser bueno.
- **Vistas por suscriptor:** caen del 42 % al 2 % (H = 55,6). Los suscriptores sobrestiman el alcance real.
- **Like rate:** sí sube con el tamaño. El like es un gesto fácil; el comentario exige más implicación, y por eso pesa más en el score.
- **Seis de las siete métricas** cambian con el tamaño, así que comparar cada canal solo con los de su tier es una necesidad.

## 5. Score T3chSeal (fase 6)

In [ ]:
# 1) Cada métrica se convierte en su PERCENTIL dentro del tier (0-100):
#    un 90 significa "mejor que el 90 % de los canales de su tamaño".
# 2) En las métricas donde MENOS es mejor (CV, irregularidad) se invierte el orden.
# 3) Media ponderada con los pesos.
pesos = {
    # métrica:        (peso, ¿más alto es mejor?)
    "comment_rate":   (0.25, True),   # comunidad: implicación real
    "like_rate":      (0.10, True),   # comunidad: implicación ligera
    "vistas_por_sub": (0.20, True),   # alcance: qué parte de los suscriptores ve el vídeo
    "mediana_vistas": (0.15, True),   # alcance: vistas del vídeo típico
    "cv_vistas":      (0.15, False),  # constancia: menos variación = más predecible
    "std_dias_entre": (0.15, False),  # constancia: menos desviación = publica más regular
}

def calcular_score(df, pesos):
    p_cols = {}
    for m, (peso, mas_es_mejor) in pesos.items():
        p_cols[f"p_{m}"] = df.groupby("tier", observed=True)[m].rank(pct=True, ascending=mas_es_mejor) * 100
    p_vals = pd.DataFrame(p_cols)
    w = pd.Series({f"p_{m}": peso for m, (peso, _) in pesos.items()})
    # Si falta una métrica (p. ej. comentarios desactivados) su peso se reparte entre las demás
    score = (p_vals * w).sum(axis=1) / p_vals.notna().mul(w).sum(axis=1)
    return p_vals, score.round(1)

def poner_sello(score):
    return pd.cut(score, [0, 50, 60, 70, 100], labels=["Sin sello", "Bronce", "Plata", "Oro"],
                  include_lowest=True)

p_vals, por_canal["score"] = calcular_score(por_canal, pesos)
por_canal[p_vals.columns] = p_vals
por_canal["sello"] = poner_sello(por_canal["score"])

print(por_canal["sello"].value_counts().sort_index())
por_canal.sort_values("score", ascending=False)[["canal", "tier", "subscribers", "score", "sello"]].head(15)

In [ ]:
# ALERTAS informativas (no cambian la nota)
por_canal["alerta_sorteos"] = por_canal["pct_sorteos"] > 0.05   # más del 5 % de vídeos con sorteo
por_canal["alerta_shorts"] = por_canal["pct_shorts"] > 0.80     # canal casi solo de Shorts

alertas = ["alerta_sorteos", "alerta_shorts"]
print(por_canal[alertas].sum())
por_canal[por_canal[alertas].any(axis=1)][["canal", "tier", "score", "sello"] + alertas]

In [ ]:
# FICHA T3chSeal de un canal (cambia el nombre para ver otro)
nombre = "GTechMx"
canal = por_canal[por_canal["canal"] == nombre].iloc[0]

print(f"📺 {canal['canal']}  |  {canal['subscribers']:,.0f} suscriptores  |  tier {canal['tier']}")
print(f"🏅 Score T3chSeal: {canal['score']} / 100  ->  {canal['sello']}\n")
for m in pesos:
    print(f"  {m:<16} valor {canal[m]:>10.3f}   percentil en su tier: {canal[f'p_{m}']:>5.1f}")
print("\n⚠️ Alertas:", [a for a in alertas if canal[a]] or "ninguna")

In [ ]:
# ¿Dónde quedan los canales conocidos?
conocidos = ["Topes de Gama", "Pro Android", "TuTecnoMundo", "clipset", "GrupoReviews", "Nate Gentile"]
por_canal[por_canal["canal"].isin(conocidos)][["canal", "tier", "subscribers", "score", "sello"]]

## 6. Validación del score

In [ ]:
# VALIDACIÓN 1: el score no debe ser un ranking de tamaño
rho, p = stats.spearmanr(por_canal["score"], por_canal["subscribers"])
print(f"Spearman score-suscriptores: rho = {rho:.2f}, p = {p:.3f}")

# Cada tier debería tener canales Oro
pd.crosstab(por_canal["tier"], por_canal["sello"])

In [ ]:
# VALIDACIÓN 2: las 3 dimensiones miden cosas distintas
# Dentro de una dimensión las métricas deberían correlacionar; entre dimensiones, poco.
cols = ["p_comment_rate", "p_like_rate", "p_vistas_por_sub", "p_mediana_vistas",
        "p_cv_vistas", "p_std_dias_entre"]
plt.figure(figsize=(7, 6))
sns.heatmap(por_canal[cols].corr(method="spearman"), annot=True, fmt=".2f",
            cmap="RdBu_r", vmin=-1, vmax=1)
plt.title("Spearman entre percentiles (comunidad | alcance | constancia)")
plt.show()

In [ ]:
# VALIDACIÓN 3: sensibilidad de los pesos
# a) Pesos IGUALES para todas las métricas
iguales = {m: (1 / len(pesos), mas) for m, (_, mas) in pesos.items()}
_, score_iguales = calcular_score(por_canal, iguales)
rho, _ = stats.spearmanr(por_canal["score"], score_iguales)
mismo = (poner_sello(score_iguales) == por_canal["sello"]).mean()
print(f"Pesos iguales: rho = {rho:.2f} | mismo sello: {mismo:.0%}")

# b) Quitando cada métrica de una en una (leave-one-out)
for quitar in pesos:
    resto = {m: v for m, v in pesos.items() if m != quitar}
    _, s = calcular_score(por_canal, resto)
    rho, _ = stats.spearmanr(por_canal["score"], s)
    print(f"Sin {quitar:<15} rho = {rho:.2f}")
# rho > 0,9 -> el ranking no depende de la elección exacta de pesos

In [ ]:
# VALIDACIÓN 4: criterio experto. Canales que considerábamos estancados antes de calcular nada
estancados = ["movilToday", "Just Unboxing", "EloYGomeZTv"]
por_canal[por_canal["canal"].isin(estancados)][["canal", "tier", "subscribers", "score", "sello"]]

## 7. Exportación para Power BI

In [ ]:
# sep=";" y decimal="," -> evita el problema de los decimales en Power BI en español
por_canal.to_csv("techseal_scores.csv", sep=";", decimal=",", index=False, encoding="utf-8-sig")

try:
    from google.colab import files
    files.download("techseal_scores.csv")
except ImportError:
    print("Guardado techseal_scores.csv")